# 學習測試(進階修改架構)2.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
!pip install fasttreeshap
!pip install shap


In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score
import seaborn as sns
import time


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備：{device}")


In [ ]:
file_level_data = []
file_level_labels = []


In [ ]:
def calculate_head_torso_angle(x_head, y_head, x_shoulder, y_shoulder):
    delta_x = x_head - x_shoulder
    delta_y = y_head - y_shoulder
    angle_radians = math.atan2(delta_y, delta_x)
    return math.degrees(angle_radians)


In [ ]:
def calculate_angular_velocity(angle_series, time_series):
    angular_velocity = np.diff(angle_series) / np.diff(time_series)
    angular_velocity = np.append(angular_velocity, 0)
    return angular_velocity


In [ ]:
def calculate_angular_velocity_change_rate(angular_velocity, time_series):
    angular_velocity_change_rate = np.diff(angular_velocity) / np.diff(time_series)
    angular_velocity_change_rate = np.append(angular_velocity_change_rate, 0)
    return angular_velocity_change_rate


In [ ]:
def calculate_angle_change(x1, y1, x2, y2, x3, y3):

    a_x = x2 - x1
    a_y = y2 - y1

    b_x = x3 - x2
    b_y = y3 - y2

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)
    if magnitude_a * magnitude_b == 0:
        return 0

    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))

    return math.acos(cos_angle) * (180 / math.pi)


In [ ]:
def calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist):

    a_x = x_elbow - x_shoulder
    a_y = y_elbow - y_shoulder

    b_x = x_wrist - x_elbow
    b_y = y_wrist - y_elbow

    dot_product = a_x * b_x + a_y * b_y

    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)

    if magnitude_a == 0 or magnitude_b == 0:
        return 0


    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))


    return math.degrees(math.acos(cos_angle))


In [ ]:
def calculate_arm_angle_for_direction(df, directions):
    arm_angles = []
    for i in range(len(directions)):
        if directions[i] == "right":

            x_shoulder = df['kpt_13'].astype(float)[i]
            y_shoulder = df['kpt_14'].astype(float)[i]
            x_elbow = df['kpt_15'].astype(float)[i]
            y_elbow = df['kpt_16'].astype(float)[i]
            x_wrist = df['kpt_17'].astype(float)[i]
            y_wrist = df['kpt_18'].astype(float)[i]
        else:

            x_shoulder = df['kpt_6'].astype(float)[i]
            y_shoulder = df['kpt_7'].astype(float)[i]
            x_elbow = df['kpt_8'].astype(float)[i]
            y_elbow = df['kpt_9'].astype(float)[i]
            x_wrist = df['kpt_10'].astype(float)[i]
            y_wrist = df['kpt_11'].astype(float)[i]


        angle = calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist)
        arm_angles.append(angle)

    return arm_angles


In [ ]:
def determine_walking_direction(df):
    """
    根據肩膀的水平位置判斷病患是向左走還是向右走，並返回行走方向。
    """
    x_shoulder_left = df['kpt_5'].astype(float)
    x_shoulder_right = df['kpt_6'].astype(float)
    directions = []
    for i in range(len(x_shoulder_left)):
        if x_shoulder_left[i] < x_shoulder_right[i]:
            directions.append("left")
        else:
            directions.append("right")
    return directions


In [ ]:
def calculate_center_of_mass_distance(df):
    center_of_mass_distances = []

    for i in range(1, len(df)):

        x_com_prev = (df['kpt_1'].iloc[i-1] + df['kpt_2'].iloc[i-1]) / 2
        y_com_prev = (df['kpt_3'].iloc[i-1] + df['kpt_4'].iloc[i-1]) / 2
        x_com_current = (df['kpt_1'].iloc[i] + df['kpt_2'].iloc[i]) / 2
        y_com_current = (df['kpt_3'].iloc[i] + df['kpt_4'].iloc[i]) / 2


        distance = math.hypot(x_com_current - x_com_prev, y_com_current - y_com_prev)
        center_of_mass_distances.append(distance)


    center_of_mass_distances.insert(0, 0)

    return center_of_mass_distances


In [ ]:
def step_x(x1, x2):
    return abs(x2 - x1)


In [ ]:
def calculate_step_length_std(step_lengths):
    return np.std(step_lengths) if len(step_lengths) > 1 else 0


In [ ]:
def calculate_acceleration(speeds, time_series):
    speeds = np.array(speeds)
    time_series = np.array(time_series)


    time_diff = np.diff(time_series)
    if np.any(time_diff == 0):
        return [0] * len(speeds)

    acceleration = np.diff(speeds) / time_diff
    acceleration = list(acceleration) + [0]
    return acceleration


In [ ]:
def calculate_hip_angle(x_shoulder, y_shoulder, x_hip, y_hip, x_knee, y_knee):

    a_x = x_shoulder - x_hip
    a_y = y_shoulder - y_hip


    b_x = x_knee - x_hip
    b_y = y_knee - y_hip


    dot_product = a_x * b_x + a_y * b_y


    magnitude_a = math.hypot(a_x, a_y)
    magnitude_b = math.hypot(b_x, b_y)


    if magnitude_a == 0 or magnitude_b == 0:
        return 0


    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))


    return math.degrees(math.acos(cos_angle))


In [ ]:
def extract_features_from_file(df):
    head_torso_angles = []
    arm_angles = []
    step_lengths = []
    accelerations = []
    shoulder_hip_distances = []
    hip_angles = []
    center_of_mass_distances = []


    previous_center_of_mass = None

    for i in range(len(df)):

        x_head = (df['kpt_1'].iloc[i] + df['kpt_2'].iloc[i]) / 2
        y_head = (df['kpt_3'].iloc[i] + df['kpt_4'].iloc[i]) / 2
        x_shoulder = (df['kpt_5'].iloc[i] + df['kpt_6'].iloc[i]) / 2
        y_shoulder = (df['kpt_7'].iloc[i] + df['kpt_8'].iloc[i]) / 2

        x_elbow = df['kpt_9'].iloc[i]
        y_elbow = df['kpt_10'].iloc[i]
        x_wrist = df['kpt_11'].iloc[i]
        y_wrist = df['kpt_12'].iloc[i]

        x_hip_left = df['kpt_11'].iloc[i]
        y_hip_left = df['kpt_12'].iloc[i]
        x_hip_right = df['kpt_13'].iloc[i]
        y_hip_right = df['kpt_14'].iloc[i]

        x_knee = df['kpt_13'].iloc[i]
        y_knee = df['kpt_14'].iloc[i]

        x_ankle_left = df['kpt_15'].iloc[i]
        x_ankle_right = df['kpt_16'].iloc[i]


        head_torso_angle = calculate_head_torso_angle(x_head, y_head, x_shoulder, y_shoulder)
        arm_angle = calculate_arm_angle(x_shoulder, y_shoulder, x_elbow, y_elbow, x_wrist, y_wrist)
        step_length = step_x(x_ankle_left, x_ankle_right)
        hip_angle = calculate_hip_angle(x_shoulder, y_shoulder, (x_hip_left + x_hip_right) / 2,
                                        (y_hip_left + y_hip_right) / 2, x_knee, y_knee)


        shoulder_hip_distance = math.hypot(x_shoulder - (x_hip_left + x_hip_right) / 2,
                                           y_shoulder - (y_hip_left + y_hip_right) / 2)


        current_center_of_mass = ((x_shoulder + (x_hip_left + x_hip_right) / 2) / 2,
                                  (y_shoulder + (y_hip_left + y_hip_right) / 2) / 2)


        if previous_center_of_mass is not None:
            com_distance = math.hypot(current_center_of_mass[0] - previous_center_of_mass[0],
                                      current_center_of_mass[1] - previous_center_of_mass[1])
            center_of_mass_distances.append(com_distance)
        else:

            center_of_mass_distances.append(0)


        previous_center_of_mass = current_center_of_mass


        head_torso_angles.append(head_torso_angle)
        arm_angles.append(arm_angle)
        step_lengths.append(step_length)
        shoulder_hip_distances.append(shoulder_hip_distance)
        hip_angles.append(hip_angle)


    time_series = np.arange(len(df))


    step_frequency = len(df) / (df.index[-1] - df.index[0] + 1)


    speeds = [step_lengths[i] / (df.index[i] + 1) for i in range(len(step_lengths))]
    accelerations = calculate_acceleration(speeds, df.index)


    acceleration_rms = np.sqrt(np.mean(np.square(accelerations)))


    hip_angular_velocity = calculate_angular_velocity(hip_angles, time_series)
    hip_angular_velocity_change_rate = calculate_angular_velocity_change_rate(hip_angular_velocity, time_series)


    features = [
        np.mean(head_torso_angles), np.std(head_torso_angles), np.max(head_torso_angles), np.min(head_torso_angles),
        np.mean(arm_angles), np.std(arm_angles), np.max(arm_angles), np.min(arm_angles),
        np.mean(step_lengths), np.std(step_lengths), np.max(step_lengths), np.min(step_lengths),
        np.mean(accelerations), np.std(accelerations), acceleration_rms,
        step_frequency,
        np.std(step_lengths) / np.mean(step_lengths) if np.mean(step_lengths) != 0 else 0,
        np.mean(shoulder_hip_distances), np.std(shoulder_hip_distances), np.max(shoulder_hip_distances), np.min(shoulder_hip_distances),
        np.mean(hip_angles), np.std(hip_angles), np.max(hip_angles), np.min(hip_angles),
        np.mean(hip_angular_velocity), np.std(hip_angular_velocity), np.max(hip_angular_velocity), np.min(hip_angular_velocity),
        np.mean(hip_angular_velocity_change_rate), np.std(hip_angular_velocity_change_rate), np.max(hip_angular_velocity_change_rate), np.min(hip_angular_velocity_change_rate),
        np.mean(center_of_mass_distances), np.std(center_of_mass_distances), np.max(center_of_mass_distances), np.min(center_of_mass_distances)
    ]


    return features


In [ ]:
def upload_data_and_label_file_level(label):
    global file_level_data, file_level_labels


    uploaded_files = files.upload()


    for file_name in uploaded_files.keys():
        df = pd.read_csv(io.BytesIO(uploaded_files[file_name]))


        if 'kpts' in df.columns:
            df['kpts'] = df['kpts'].str.replace('"', '').str.strip()
            kpts_columns = df['kpts'].str.split(',', expand=True)
            for i in range(kpts_columns.shape[1]):
                kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)
            df = df.join(kpts_columns, rsuffix='_kpts')
            for col in kpts_columns.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)


        features = extract_features_from_file(df)


        file_level_data.append(features)
        file_level_labels.append(label)

        print(f"已處理文件 {file_name}，標籤為 {label}。")


In [ ]:
print("上傳第一批數據並指定標籤為1")
upload_data_and_label_file_level(1)

print("上傳第二批數據並指定標籤為2")
upload_data_and_label_file_level(2)

print("上傳第三批數據並指定標籤為3")
upload_data_and_label_file_level(3)

print("上傳第四批數據並指定標籤為4")
upload_data_and_label_file_level(4)


處理

In [ ]:
file_level_labels = [label - 1 for label in file_level_labels]


In [ ]:
print(f"標籤範圍: {min(file_level_labels)} 到 {max(file_level_labels)}")


自動執行HFS流程(混合基因演算法與神經網路)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler


In [ ]:
def perform_hfs(X, y):


    correlation_matrix = np.corrcoef(X, rowvar=False)


    low_correlation_features = [i for i in range(X.shape[1]) if np.abs(np.corrcoef(X[:, i], y)[0, 1]) < 0.3]
    X_filtered = np.delete(X, low_correlation_features, axis=1)



    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rfe = RFE(estimator=rf, n_features_to_select=20)
    rfe.fit(X_filtered, y)
    X_rfe = X_filtered[:, rfe.support_]


    lasso = Lasso(alpha=0.01)
    lasso.fit(X_rfe, y)
    important_features = np.where(np.abs(lasso.coef_) > 0)[0]
    X_final = X_rfe[:, important_features]


    return X_final


In [ ]:
def update_features_using_hfs():
    global file_level_data, file_level_labels


    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(file_level_data)


    selected_features = perform_hfs(features_scaled, file_level_labels)


    file_level_data = selected_features


In [ ]:
if np.any(np.isnan(file_level_data)) or np.any(np.isinf(file_level_data)):
    print("數據中包含 NaN 或 Inf 值，需要進行處理。")


In [ ]:
update_features_using_hfs()


In [ ]:
correlation_matrix = np.corrcoef(file_level_data, rowvar=False)


In [ ]:
print("進行混合特徵選擇（HFS）")
update_features_using_hfs()


In [ ]:
print(f"目前的特徵數量：{file_level_data.shape[1]}")


監督式學習

In [ ]:
class SeverityClassifier(nn.Module):
    def __init__(self, input_size, num_classes=4):
        super(SeverityClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.2)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.2)

        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(p=0.2)

        self.fc4 = nn.Linear(64, 32)
        self.bn4 = nn.BatchNorm1d(32)
        self.relu4 = nn.ReLU()
        self.dropout4 = nn.Dropout(p=0.2)

        self.fc5 = nn.Linear(32, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.fc2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        out = self.fc3(out)
        out = self.bn3(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        out = self.fc4(out)
        out = self.bn4(out)
        out = self.relu4(out)
        out = self.dropout4(out)

        out = self.fc5(out)
        return out


In [ ]:
class PatientDataset(Dataset):
    def __init__(self, data, labels):
        self.features = data
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print("早停法觸發")
                return True
        return False


In [ ]:
file_level_data = np.array(file_level_data)
file_level_labels = np.array(file_level_labels)


In [ ]:
features = file_level_data
labels = file_level_labels


In [ ]:
scaler = StandardScaler()
features = scaler.fit_transform(features)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.3, random_state=42, stratify=labels)


X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)


In [ ]:
train_dataset = PatientDataset(X_train, y_train)
test_dataset = PatientDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
from torch.utils.data import random_split


val_size = int(0.2 * len(train_dataset))
train_size = len(train_dataset) - val_size


train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])


val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [ ]:
input_size = features.shape[1]
num_classes = 4
model = SeverityClassifier(input_size=input_size, num_classes=num_classes).to(device)



criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001 , weight_decay=1e-5)


In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau


In [ ]:
early_stopping = EarlyStopping(patience=10, min_delta=0.001)


def train_model(num_epochs=200):
    model.train()
    start_time = time.time()
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


        val_loss = 0.0
        model.eval()
        with torch.no_grad():
            for val_features, val_labels in val_loader:
                val_features = val_features.to(device)
                val_labels = val_labels.to(device)
                val_outputs = model(val_features)
                val_loss += criterion(val_outputs, val_labels).item()
        val_loss /= len(val_loader)


        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Val Loss: {val_loss:.4f}')


        if early_stopping.step(val_loss):
            break

    end_time = time.time()
    total_time = end_time - start_time
    print(f'模型訓練總共花費的時間: {total_time:.2f} 秒')


In [ ]:
def test_model():
    model.eval()
    start_time = time.time()
    true_labels = []
    predicted_labels = []
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)

            true_labels.extend(labels.cpu().numpy())
            predicted_labels.extend(predicted.cpu().numpy())

    end_time = time.time()
    total_time = end_time - start_time


    accuracy = 100 * np.sum(np.array(true_labels) == np.array(predicted_labels)) / len(true_labels)
    print(f'測試集上的準確率: {accuracy:.2f}%')


    return true_labels, predicted_labels


In [ ]:
train_model(num_epochs=200)


true_labels, predicted_labels = test_model()


In [ ]:
def test_model_with_thresholds():
    model.eval()
    start_time = time.time()
    true_labels = []
    predicted_labels = []
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)


            outputs = model(features)
            probabilities = torch.softmax(outputs, dim=1)


            max_probs, predicted = torch.max(probabilities, 1)

            true_labels.extend(labels.cpu().numpy())
            predicted_labels.append((max_probs.cpu().numpy(), predicted.cpu().numpy()))

    end_time = time.time()
    total_time = end_time - start_time

    print(f'模型測試總共花費的時間: {total_time:.2f} 秒')


    thresholds = np.arange(0.01, 1.01, 0.01)
    for threshold in thresholds:

        adjusted_predictions = [
            pred if prob >= threshold else -1
            for (probs, preds) in predicted_labels
            for prob, pred in zip(probs, preds)
        ]


        valid_indices = [i for i, pred in enumerate(adjusted_predictions) if pred != -1]
        filtered_true_labels = [true_labels[i] for i in valid_indices]
        filtered_predictions = [adjusted_predictions[i] for i in valid_indices]


        if len(filtered_true_labels) == 0:
            print(f'閾值 {threshold:.2f} 下沒有有效的樣本。')
            continue


        accuracy = 100 * np.sum(np.array(filtered_true_labels) == np.array(filtered_predictions)) / len(filtered_true_labels)
        print(f'閾值 {threshold:.2f} 下的準確率: {accuracy:.2f}%')

test_model_with_thresholds()


In [ ]:
def evaluate_model(true_labels, predicted_labels):

    precision = precision_score(true_labels, predicted_labels, average='weighted')
    recall = recall_score(true_labels, predicted_labels, average='weighted')
    f1 = f1_score(true_labels, predicted_labels, average='weighted')


    print(f'精確率（Precision）: {precision:.2f}')
    print(f'召回率（Recall）: {recall:.2f}')
    print(f'F1 分數: {f1:.2f}')


In [ ]:
true_labels, predicted_labels = test_model()


In [ ]:
evaluate_model(true_labels, predicted_labels)


In [ ]:
def plot_confusion_matrix(true_labels, predicted_labels):

    cm = confusion_matrix(true_labels, predicted_labels)


    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title('Confusion Matrix')
    plt.show()


In [ ]:
plot_confusion_matrix(true_labels, predicted_labels)


In [ ]:
import shap
import fasttreeshap

from sklearn.ensemble import RandomForestClassifier


In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)


In [ ]:
explainer = fasttreeshap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)


In [ ]:
feature_names = [
    'mean_head_torso_angle', 'std_head_torso_angle', 'max_head_torso_angle', 'min_head_torso_angle',
    'mean_arm_angle', 'std_arm_angle', 'max_arm_angle', 'min_arm_angle',
    'mean_step_length', 'std_step_length', 'max_step_length', 'min_step_length',
    'mean_acceleration', 'std_acceleration', 'acceleration_rms',
    'step_frequency',
    'step_length_variation_coefficient',
    'mean_shoulder_hip_distance', 'std_shoulder_hip_distance', 'max_shoulder_hip_distance', 'min_shoulder_hip_distance',
    'mean_hip_angle', 'std_hip_angle', 'max_hip_angle', 'min_hip_angle',
    'mean_hip_angular_velocity', 'std_hip_angular_velocity', 'max_hip_angular_velocity', 'min_hip_angular_velocity',
    'mean_hip_angular_velocity_change_rate', 'std_hip_angular_velocity_change_rate', 'max_hip_angular_velocity_change_rate', 'min_hip_angular_velocity_change_rate',
    'mean_center_of_mass_distance', 'std_center_of_mass_distance', 'max_center_of_mass_distance', 'min_center_of_mass_distance'
]


In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=feature_names)


測試


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix



y_true = [0, 1, 2, 3, 1, 0, 2, 3, 0, 1, 2, 3]
y_pred = [0, 1, 1, 3, 1, 0, 2, 2, 0, 0, 2, 3]


cm = confusion_matrix(y_true, y_pred)


plt.figure(figsize=(10, 8))
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Class 0', 'Class 1', 'Class 2', 'Class 3'], yticklabels=['Class 0', 'Class 1', 'Class 2', 'Class 3'])
ax.set_xlabel('預測標籤')
ax.set_ylabel('實際標籤')


for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if i == j:
            ax.text(j + 0.5, i + 0.5, 'True Positive\n預測正確', ha='center', va='center', color='green', fontsize=10, weight='bold')
        else:
            ax.text(j + 0.5, i + 0.5, 'False Prediction\n預測錯誤', ha='center', va='center', color='red', fontsize=10, weight='bold')

plt.title('Confusion Matrix for Four-Class Prediction')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
feature_names = [
    'mean_head_torso_angle', 'std_head_torso_angle', 'max_head_torso_angle', 'min_head_torso_angle',
    'mean_arm_angle', 'std_arm_angle', 'max_arm_angle', 'min_arm_angle',
    'mean_step_length', 'std_step_length', 'max_step_length', 'min_step_length',
    'mean_acceleration', 'std_acceleration', 'acceleration_rms',
    'step_frequency',
    'step_length_variation_coefficient',
    'mean_shoulder_hip_distance', 'std_shoulder_hip_distance', 'max_shoulder_hip_distance', 'min_shoulder_hip_distance',
    'mean_hip_angle', 'std_hip_angle', 'max_hip_angle', 'min_hip_angle',
    'mean_hip_angular_velocity', 'std_hip_angular_velocity', 'max_hip_angular_velocity', 'min_hip_angular_velocity',
    'mean_hip_angular_velocity_change_rate', 'std_hip_angular_velocity_change_rate', 'max_hip_angular_velocity_change_rate', 'min_hip_angular_velocity_change_rate'
]


In [ ]:
df_features = pd.DataFrame(file_level_data, columns=feature_names)


In [ ]:
correlation_matrix = df_features.corr()


In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.show()


繪製 ROC 曲線

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score


In [ ]:
def plot_roc_pr_curves():
    model.eval()
    true_labels = []
    predicted_probs = []
    with torch.no_grad():
        for features, labels in test_loader:
            features = features.to(device)
            labels = labels.to(device)


            outputs = model(features)
            probabilities = torch.softmax(outputs, dim=1)
            predicted_probs.extend(probabilities.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    true_labels = np.array(true_labels)
    predicted_probs = np.array(predicted_probs)
    num_classes = predicted_probs.shape[1]


    for i in range(num_classes):

        fpr, tpr, _ = roc_curve(true_labels == i, predicted_probs[:, i])
        roc_auc = auc(fpr, tpr)


        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve for Class {i}')
        plt.legend(loc='lower right')
        plt.show()


        precision, recall, _ = precision_recall_curve(true_labels == i, predicted_probs[:, i])
        avg_precision = average_precision_score(true_labels == i, predicted_probs[:, i])


        plt.figure()
        plt.plot(recall, precision, color='b', lw=2, label=f'Average Precision = {avg_precision:.2f}')
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title(f'Precision-Recall Curve for Class {i}')
        plt.legend(loc='lower left')
        plt.show()


In [ ]:
plot_roc_pr_curves()
